# Pulpermill development tools (Kaggle)
For engine measurements and prompt trials on the real model. It does not start the server: to run Pulpermill, use `pulpermill_backend.ipynb`.

Set the accelerator to **GPU T4 x2** and **Internet = on**, then run the first four steps (settings, install, check, CUDA) as in the launcher, and use the tools below. No secrets are needed here; `HF_TOKEN` makes the model download faster.

## Settings

In [ ]:
BRANCH = "main"        # which version of the code to use: a branch or a release tag of the repository

## 1. Get the code and install

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/LacChe/pulpermill"
DIR = "/kaggle/working/pulpermill"

# The code was already imported in this session (an earlier run): a new clone would not replace it, so stop here.
if "pulpermill" in sys.modules:
    raise RuntimeError("The code was already imported in this session. Stop the session and run all the cells again, so the fresh clone is the one that runs.")

os.chdir("/kaggle/working")
shutil.rmtree(DIR, ignore_errors=True)         # a fresh clone every session: an old copy would keep running the old code
try:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DIR], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(f"Could not clone {REPO} (branch or tag {BRANCH!r}). Check that Internet is on and that BRANCH names a real branch or tag.") from error
os.chdir(DIR)
sys.path.insert(0, DIR)
print("running commit", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())

# The server's own packages. llama-cpp-python comes in step 3, with CUDA.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "pulpermill/backend/requirements.txt"], check=True)

## 2. Check the session

In [ ]:
from pulpermill.backend.kaggle_setup import check_session
check_session(required_secrets=())      # no secrets are needed here

## 3. llama-cpp-python with CUDA
Loads the CUDA 12 runtime libraries (Kaggle's image may carry only CUDA 13), then uses the llama-cpp-python that is installed if it loads and can use the GPU, and otherwise installs a ready-made CUDA 12 build. It must be run again after a restart. The first run takes a few minutes.

In [ ]:
from pulpermill.backend.kaggle_setup import ensure_cuda_llama
ensure_cuda_llama()

## Measuring (run once, in a separate session or after stopping the launcher)
Fills in `docs/backend-notes.md`: tokenizer parity, speed, cancellation, memory. Copy the printed Markdown back.

In [ ]:
from dataclasses import replace
from pulpermill.backend.config import Config
from pulpermill.backend.engine import download
from pulpermill.backend.engine.llama import load
from tools.measure_engine import measure_all, memory_by_ctx
model = Config().model
path = download.ensure_model(model)
engine = load(model, path)
measure_all(engine)

## Trials on the real model (for the prompts work)
Runs a chapter loop (the old `tools/run_trials.py` is at the git tag `pre-prompt-design`) in this session's engine, with no server. Needs the engine loaded as in the measuring cell. Each call's sampling and token/second record is kept in `rows`.

In [ ]:
from pulpermill.backend.complete import make_complete
from pulpermill.backend.config import Config

rows = []
complete = make_complete(engine, max_tokens=1600, params=Config().prose, record=rows)
# request = <a generate request dict, e.g. from evals/trials>
# text = complete(prompt)  # prompt design supplies the trial loop

## The run log
One line per call (settings first), from a server run in this session. Paste the lines you want checked.

In [ ]:
import json
for line in open("/kaggle/working/pulpermill-runs.jsonl").read().splitlines()[-20:]:
    r = json.loads(line)
    print(r["record"], r.get("call", ""), r.get("result", ""), r.get("seconds", ""), "retries", r.get("retries", ""), "squeeze", r.get("squeeze", ""))

## Memory by context size
Frees the engine, then measures memory at several context sizes.

In [ ]:
del engine
memory_by_ctx(model, path, [8192, 12288, 16384])